In [2]:
import os
import sys
import datetime
import pandas as pd
from upbit.client import Upbit
import time
import traceback

# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
upper_dir = "/home/trade_core"
sys.path.append(upper_dir)
from loggers.logger import KimpBotLogger

In [8]:
upbit_access_key = "x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8"
upbit_secret_key = "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL"

In [16]:
class UserUpbitAdaptor:
    def __init__(self, logging_dir=None):
        self.user_client_dict = {}
        self.retry_term_sec = 0.2
        self.retry_count_limit = 2
        self.upbit_plug_logger = KimpBotLogger("user_upbit_plug", logging_dir).logger
        self.upbit_plug_logger.info(f"user_upbit_plug_logger started.")

    def load_user_client(self, access_key, secret_key):
        user_client = self.user_client_dict.get(access_key)
        if user_client is None:
            self.user_client_dict[access_key] = Upbit(access_key, secret_key)
            return self.user_client_dict[access_key]
        else:
            return user_client
        
    def get_spot_balance(self, access_key, secret_key, return_dict=None):
        upbit_client = self.load_user_client(access_key, secret_key)
        result_df = pd.DataFrame(upbit_client.Account.Account_info()['result'])
        result_df.loc[:, ['balance','locked','avg_buy_price']] = result_df[['balance','locked','avg_buy_price']].astype(float)
        result_df = result_df.rename(columns={'currency':'asset', 'balance':'free', 'locked':'used'})
        if return_dict is None:
            return result_df
        else:
            return_dict['res'] = result_df

    def get_balance(self, access_key, secret_key, market_type='SPOT'):
        if market_type == "SPOT":
            return self.get_spot_balance(access_key, secret_key)
        elif market_type == "USD_M":
            raise Exception(f"market_type: {market_type} is not supported yet.")
        elif market_type == "COIN_M":
            raise Exception(f"market_type: {market_type} is not supported yet.")
        else:
            raise Exception(f"Invalid market_type: {market_type}")
    
    def check_api_key(self, access_key, secret_key, futures=False):
        self.user_client_dict.pop(access_key, None)
        try:
            if futures is False:
                self.get_spot_balance(access_key, secret_key)
            else:
                # self.get_coinm_balance(access_key, secret_key)
                raise Exception(f"futures market is not supported yet.")
            return (True, 'OK')
        except Exception as e:
            self.user_client_dict.pop(access_key, None)
            return (False, str(e))

In [17]:
user_upbit_adaptor = UserUpbitAdaptor()

2024-02-15 07:48:12,855 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-15 07:48:12,855 - user_upbit_plug - INFO - user_upbit_plug_logger started.


In [25]:
balance_df = user_upbit_adaptor.get_spot_balance(upbit_access_key, upbit_secret_key)
balance_df

,asset,free,locked,avg_buy_price,avg_buy_price_modified,unit_currency
0,KRW,42679379.678397,0.0,0.0,True,KRW
1,ETH,0.0,0.0,2958528.427637,False,KRW
2,XRP,11321.0,0.0,704.829091,False,KRW
3,ETC,311.6,0.0,31713.536244,False,KRW
4,WAVES,0.0,0.0,2521.220004,False,KRW
5,LSK,2442.0,0.0,2037.672204,False,KRW
6,ARK,0.0,0.0,2065.0,False,KRW
7,BCH,0.0,0.0,340470.959448,False,KRW
8,STORJ,0.0,0.0,935.25234,False,KRW
9,USDT,0.058399,0.0,1079.359474,False,KRW
